# Bold Sakura Analysis - December 2025 Performance Evaluation

## Objective
Evaluate Bold brands Sakura performance and establish next year Bold Sakura launch strategy.

## Background
- December 2025: Bold Sakura launched
- Initiative objective: Trial user acquisition
- Bold Gel Ball Sakura performed well in 2024, so it was expanded to Bold Gel

## Analysis Agenda
1. **Total Bold December Status**: Value IYA, Unit IYA, Value Share (National & Channel)
2. **Bold Sakura Contribution**: How much Sakura contributed to Bold performance
3. **SoV Analysis**: Compare YA Bold Sakura vs TY Bold Sakura (trial user acquisition)
4. **Store Level Analysis**: Distribution pattern analysis (Gel Ball only vs Gel Sakura stores)

## Hypothesis
- H1: Total Bold brand (Gel + Gel Ball) is flat vs YA because Bold Gel is underperforming
- H2: Bold Gel acquired unique users (SoV) compared to 2024 → Bold Gel was successful despite flat value share vs YA

## Product Information
### Bold Sakura GTIN Codes
| Product | GTIN |
|---------|------|
| Bold Gel Ball Sakura | 04987176344250 |
| Bold Gel Ball Sakura | 04987176344229 |
| Bold Gel Ball Sakura | 04987176344236 |
| Bold Gel Sakura | 04987176349668 |
| Bold Gel Sakura | 04987176349682 |
| Bold Gel Sakura | 04987176349712 |

---
**Analysis Date:** 2026/01/20
**Analyst:** Sugimoto

## 1. Setup and Configuration

In [1]:
# Import required libraries
from databricks import sql
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.0f}' if abs(x) >= 1 else f'{x:.2%}')

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [2]:
# Load environment variables from parent directory
load_dotenv(dotenv_path='../../.env')

# Validate credentials
required_vars = ['DATABRICKS_HOST', 'DATABRICKS_HTTP_PATH', 'DATABRICKS_TOKEN']
missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

print("✓ Environment configured")

✓ Environment configured


## 2. Analysis Parameters

In [19]:
# =============================================================================
# ANALYSIS PARAMETERS
# =============================================================================

# Customer Filter (National = use all customers)
# Available customers: cds_8005 (TSURUHA), cds_8008 (KOHNAN), cds_8010 (TRIAL), etc.
customer_filters = ['cds_8005', 'cds_8008', 'cds_8010', 'cds_8011', 'cds_8012', 'cds_8013']

# Time Period (December Analysis)
ty_start_date = '2025-12-01'
ty_end_date = '2025-12-31'
ya_start_date = '2024-12-01'
ya_end_date = '2024-12-31'

# Pre-Sakura period (3 months before for baseline comparison)
pre_sakura_start = '2025-09-01'
pre_sakura_end = '2025-11-30'

# Category Filter
category_filter = 'Laundry'

# Product Hierarchy
# Bold products - Sub-brand names (half-width katakana)
bold_gel_ball_sub_brand = 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ'  # Bold Gel Ball
bold_gel_sub_brand = 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙ'  # Bold Gel
bold_sub_brands = [bold_gel_ball_sub_brand, bold_gel_sub_brand]

# Bold Sakura GTIN codes (with leading zeros as they appear in database)
bold_gel_ball_sakura_gtins = ['04987176344250', '04987176344229', '04987176344236']
bold_gel_sakura_gtins = ['04987176349668', '04987176349682', '04987176349712']
all_sakura_gtins = bold_gel_ball_sakura_gtins + bold_gel_sakura_gtins

print("✓ Parameters configured")
print(f"  TY Period: {ty_start_date} to {ty_end_date}")
print(f"  YA Period: {ya_start_date} to {ya_end_date}")
print(f"  Pre-Sakura Period: {pre_sakura_start} to {pre_sakura_end}")
print(f"  Category: {category_filter}")
print(f"  Bold Sub-brands: {bold_sub_brands}")
print(f"  Sakura GTINs: {len(all_sakura_gtins)} products")

✓ Parameters configured
  TY Period: 2025-12-01 to 2025-12-31
  YA Period: 2024-12-01 to 2024-12-31
  Pre-Sakura Period: 2025-09-01 to 2025-11-30
  Category: Laundry
  Bold Sub-brands: ['ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ', 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙ']
  Sakura GTINs: 6 products


In [4]:
# Database connection helper function
def execute_query(query):
    """Execute SQL query and return DataFrame"""
    with sql.connect(
        server_hostname=os.getenv("DATABRICKS_HOST"),
        http_path=os.getenv("DATABRICKS_HTTP_PATH"),
        access_token=os.getenv("DATABRICKS_TOKEN")
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            result = cursor.fetchall()
            columns = [desc[0] for desc in cursor.description]
            return pd.DataFrame(result, columns=columns)

print("✓ Query helper function defined")

✓ Query helper function defined


---
## 3. Analysis 1: Total Bold December Status

### Objective
- Value IYA (vs Year Ago)
- Unit IYA
- Value Share - National & Channel level

In [20]:
# Build query for Bold brand performance (National)
query_bold_status = f"""
WITH base AS (
    SELECT
        idpos.data_provider_code_part AS customer,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        prod.jp_category_name,
        prod.jp_brand_alter_lang_name AS brand,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        prod.jp_item_gtin AS gtin,
        prod.jp_prod_alter_lang_name AS product_name,
        cust.jp_cust_channel_name AS channel,
        idpos.pos_sales_amt AS value,
        idpos.pos_unit_sales_qty AS unit
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.site_dim_ext_vw site_ext ON idpos.site_key = site_ext.site_key
    LEFT JOIN id_pos_ai_1.site_dim_vw site ON idpos.site_key = site.site_key
    LEFT JOIN id_pos_ai_1.cust_dim_ext_vw cust ON site.own_party_key = cust.cust_key
    WHERE prod.jp_category_name = '{category_filter}'
        AND idpos.sales_period_group_end_date_part BETWEEN '{ya_start_date}' AND '{ty_end_date}'
),

-- TY December Bold performance
ty_bold AS (
    SELECT
        'National' AS level,
        'Total' AS channel,
        sub_brand,
        CASE 
            WHEN gtin IN ({','.join(["'" + g + "'" for g in all_sakura_gtins])}) THEN 'Sakura'
            ELSE 'Regular'
        END AS product_type,
        SUM(value) AS ty_value,
        SUM(unit) AS ty_unit
    FROM base
    WHERE sub_brand IN ('{bold_gel_ball_sub_brand}', '{bold_gel_sub_brand}')
        AND purchase_date BETWEEN '{ty_start_date}' AND '{ty_end_date}'
    GROUP BY sub_brand, 
        CASE WHEN gtin IN ({','.join(["'" + g + "'" for g in all_sakura_gtins])}) THEN 'Sakura' ELSE 'Regular' END
),

-- YA December Bold performance  
ya_bold AS (
    SELECT
        'National' AS level,
        'Total' AS channel,
        sub_brand,
        CASE 
            WHEN gtin IN ({','.join(["'" + g + "'" for g in bold_gel_ball_sakura_gtins])}) THEN 'Sakura'
            ELSE 'Regular'
        END AS product_type,
        SUM(value) AS ya_value,
        SUM(unit) AS ya_unit
    FROM base
    WHERE sub_brand IN ('{bold_gel_ball_sub_brand}', '{bold_gel_sub_brand}')
        AND purchase_date BETWEEN '{ya_start_date}' AND '{ya_end_date}'
    GROUP BY sub_brand,
        CASE WHEN gtin IN ({','.join(["'" + g + "'" for g in bold_gel_ball_sakura_gtins])}) THEN 'Sakura' ELSE 'Regular' END
),

-- Category total for share calculation
ty_category AS (
    SELECT SUM(value) AS ty_category_value
    FROM base
    WHERE purchase_date BETWEEN '{ty_start_date}' AND '{ty_end_date}'
),
ya_category AS (
    SELECT SUM(value) AS ya_category_value
    FROM base
    WHERE purchase_date BETWEEN '{ya_start_date}' AND '{ya_end_date}'
)

SELECT
    COALESCE(ty.sub_brand, ya.sub_brand) AS sub_brand,
    COALESCE(ty.product_type, ya.product_type) AS product_type,
    COALESCE(ty.ty_value, 0) AS ty_value,
    COALESCE(ya.ya_value, 0) AS ya_value,
    CASE WHEN COALESCE(ya.ya_value, 0) > 0 
         THEN (COALESCE(ty.ty_value, 0) / ya.ya_value) * 100 
         ELSE NULL END AS value_IYA,
    COALESCE(ty.ty_unit, 0) AS ty_unit,
    COALESCE(ya.ya_unit, 0) AS ya_unit,
    CASE WHEN COALESCE(ya.ya_unit, 0) > 0 
         THEN (COALESCE(ty.ty_unit, 0) / ya.ya_unit) * 100 
         ELSE NULL END AS unit_IYA,
    ROUND(COALESCE(ty.ty_value, 0) / tc.ty_category_value * 100, 2) AS ty_value_share,
    ROUND(COALESCE(ya.ya_value, 0) / yc.ya_category_value * 100, 2) AS ya_value_share
FROM ty_bold ty
FULL OUTER JOIN ya_bold ya 
    ON ty.sub_brand = ya.sub_brand 
    AND ty.product_type = ya.product_type
CROSS JOIN ty_category tc
CROSS JOIN ya_category yc
ORDER BY sub_brand, product_type
"""

print("✓ Bold status query built")
print(f"  Query length: {len(query_bold_status)} characters")

✓ Bold status query built
  Query length: 3690 characters


In [21]:
# Execute Bold status query
df_bold_status = execute_query(query_bold_status)

# Convert numeric columns
numeric_cols = ['ty_value', 'ya_value', 'value_IYA', 'ty_unit', 'ya_unit', 'unit_IYA', 'ty_value_share', 'ya_value_share']
for col in numeric_cols:
    df_bold_status[col] = pd.to_numeric(df_bold_status[col], errors='coerce')

print(f"✓ Retrieved {len(df_bold_status)} rows")
df_bold_status

✓ Retrieved 4 rows


,sub_brand,product_type,ty_value,ya_value,value_IYA,ty_unit,ya_unit,unit_IYA,ty_value_share,ya_value_share
0,ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,Regular,"245,003,452","280,877,781",87,"409,156","474,233",86,3,3
1,ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,Sakura,"8,315,716",0.00%,NaN,"35,749",0.00%,NaN,10.00%,0.00%
2,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,Regular,"537,108,128","575,343,769",93,"404,474","434,684",93,7,7
3,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,Sakura,"5,588,311",0.00%,NaN,"16,490",0.00%,NaN,7.00%,0.00%


In [22]:
# Create summary table for Bold brand performance
# Aggregate by sub-brand
df_bold_summary = df_bold_status.groupby('sub_brand').agg({
    'ty_value': 'sum',
    'ya_value': 'sum',
    'ty_unit': 'sum',
    'ya_unit': 'sum',
    'ty_value_share': 'sum',
    'ya_value_share': 'sum'
}).reset_index()

# Calculate IYA
df_bold_summary['value_IYA'] = (df_bold_summary['ty_value'] / df_bold_summary['ya_value'] * 100).round(1)
df_bold_summary['unit_IYA'] = (df_bold_summary['ty_unit'] / df_bold_summary['ya_unit'] * 100).round(1)
df_bold_summary['share_pt_chg'] = (df_bold_summary['ty_value_share'] - df_bold_summary['ya_value_share']).round(2)

# Add Total Bold row
total_row = pd.DataFrame([{
    'sub_brand': 'Total Bold',
    'ty_value': df_bold_summary['ty_value'].sum(),
    'ya_value': df_bold_summary['ya_value'].sum(),
    'ty_unit': df_bold_summary['ty_unit'].sum(),
    'ya_unit': df_bold_summary['ya_unit'].sum(),
    'ty_value_share': df_bold_summary['ty_value_share'].sum(),
    'ya_value_share': df_bold_summary['ya_value_share'].sum()
}])
total_row['value_IYA'] = (total_row['ty_value'] / total_row['ya_value'] * 100).round(1)
total_row['unit_IYA'] = (total_row['ty_unit'] / total_row['ya_unit'] * 100).round(1)
total_row['share_pt_chg'] = (total_row['ty_value_share'] - total_row['ya_value_share']).round(2)

df_bold_summary = pd.concat([df_bold_summary, total_row], ignore_index=True)

print("\n📊 Bold Brand December Performance Summary")
print("=" * 80)
df_bold_summary


📊 Bold Brand December Performance Summary


,sub_brand,ty_value,ya_value,ty_unit,ya_unit,ty_value_share,ya_value_share,value_IYA,unit_IYA,share_pt_chg
0,ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,"253,319,168","280,877,781","444,905","474,233",3,3,90,94,-24.00%
1,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,"542,696,439","575,343,769","420,964","434,684",7,7,94,97,-19.00%
2,Total Bold,"796,015,607","856,221,550","865,869","908,917",10,10,93,95,-43.00%


In [23]:
# Visualize Bold performance
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=('Value IYA by Sub-brand', 'Value Share Comparison'),
                    specs=[[{'type': 'bar'}, {'type': 'bar'}]])

# IYA Chart
colors = ['#1f77b4' if x >= 100 else '#d62728' for x in df_bold_summary['value_IYA']]
fig.add_trace(
    go.Bar(x=df_bold_summary['sub_brand'], y=df_bold_summary['value_IYA'],
           marker_color=colors, text=df_bold_summary['value_IYA'].apply(lambda x: f'{x:.1f}%'),
           textposition='outside', name='Value IYA'),
    row=1, col=1
)
fig.add_hline(y=100, line_dash='dash', line_color='gray', row=1, col=1)

# Value Share Chart
fig.add_trace(
    go.Bar(x=df_bold_summary['sub_brand'], y=df_bold_summary['ty_value_share'],
           name='TY Share', marker_color='#2ecc71'),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=df_bold_summary['sub_brand'], y=df_bold_summary['ya_value_share'],
           name='YA Share', marker_color='#95a5a6'),
    row=1, col=2
)

fig.update_layout(title='Bold Brand December 2025 Performance', height=400, showlegend=True)
fig.show()

---
## 4. Analysis 2: Bold Sakura Contribution

### Objective
- How much did Bold Sakura contribute to total Bold performance?
- Compare Gel Ball Sakura vs Gel Sakura contribution

In [24]:
# Calculate Sakura contribution from existing data
df_sakura = df_bold_status[df_bold_status['product_type'] == 'Sakura'].copy()
df_regular = df_bold_status[df_bold_status['product_type'] == 'Regular'].copy()

# Sakura contribution table
print("\n🌸 Bold Sakura Contribution Analysis")
print("=" * 80)

total_ty_value = df_bold_status['ty_value'].sum()
total_ya_value = df_bold_status['ya_value'].sum()

sakura_ty_value = df_sakura['ty_value'].sum()
sakura_ya_value = df_sakura['ya_value'].sum()

print(f"\nTotal Bold TY Value: ¥{total_ty_value:,.0f}")
print(f"Total Bold YA Value: ¥{total_ya_value:,.0f}")
print(f"\nSakura TY Value: ¥{sakura_ty_value:,.0f} ({sakura_ty_value/total_ty_value*100:.1f}% of Bold)")
print(f"Sakura YA Value: ¥{sakura_ya_value:,.0f} ({sakura_ya_value/total_ya_value*100:.1f}% of Bold)")

# By sub-brand
print("\n📦 Sakura Contribution by Sub-brand:")
df_sakura_summary = df_sakura.copy()
df_sakura_summary['contribution_ty'] = (df_sakura_summary['ty_value'] / total_ty_value * 100).round(1)
df_sakura_summary['contribution_ya'] = (df_sakura_summary['ya_value'] / total_ya_value * 100).round(1)
df_sakura_summary


🌸 Bold Sakura Contribution Analysis

Total Bold TY Value: ¥796,015,607
Total Bold YA Value: ¥856,221,550

Sakura TY Value: ¥13,904,027 (1.7% of Bold)
Sakura YA Value: ¥0 (0.0% of Bold)

📦 Sakura Contribution by Sub-brand:


,sub_brand,product_type,ty_value,ya_value,value_IYA,ty_unit,ya_unit,unit_IYA,ty_value_share,ya_value_share,contribution_ty,contribution_ya
1,ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,Sakura,"8,315,716",0.00%,NaN,"35,749",0.00%,NaN,10.00%,0.00%,1,0.00%
3,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,Sakura,"5,588,311",0.00%,NaN,"16,490",0.00%,NaN,7.00%,0.00%,70.00%,0.00%


In [25]:
# Visualize Sakura contribution
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('TY Sakura vs Regular', 'Sakura Contribution by Sub-brand'),
                    specs=[[{'type': 'pie'}, {'type': 'bar'}]])

# Pie chart - TY composition
fig.add_trace(
    go.Pie(labels=df_bold_status['product_type'] + ' (' + df_bold_status['sub_brand'].str[-10:] + ')',
           values=df_bold_status['ty_value'],
           hole=0.4,
           marker_colors=['#ff9ff3', '#ff6b6b', '#54a0ff', '#5f27cd']),
    row=1, col=1
)

# Bar chart - Sakura by sub-brand
if len(df_sakura) > 0:
    fig.add_trace(
        go.Bar(x=df_sakura['sub_brand'], y=df_sakura['ty_value'],
               name='TY Sakura', marker_color='#ff9ff3',
               text=df_sakura['ty_value'].apply(lambda x: f'¥{x/1000000:.1f}M')),
        row=1, col=2
    )
    fig.add_trace(
        go.Bar(x=df_sakura['sub_brand'], y=df_sakura['ya_value'],
               name='YA Sakura', marker_color='#95a5a6'),
        row=1, col=2
    )

fig.update_layout(title='Bold Sakura Contribution December 2025', height=400)
fig.show()

### 📈 Pre-Sakura Baseline Comparison (3-Month Analysis)

**Objective:** Compare Bold value share in pre-Sakura period (Sep-Nov 2025) vs Sakura period (Dec 2025) to assess if Sakura provided additional uplift.

In [26]:
# Build query to compare pre-Sakura vs Sakura period Bold value share
query_pre_sakura_comparison = f"""
WITH base AS (
    SELECT
        idpos.data_provider_code_part AS customer,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        prod.jp_category_name,
        prod.jp_brand_alter_lang_name AS brand,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        prod.jp_item_gtin AS gtin,
        idpos.pos_sales_amt AS value,
        idpos.pos_unit_sales_qty AS unit,
        CASE 
            WHEN CAST(idpos.sales_period_group_end_date_part AS DATE) BETWEEN '{pre_sakura_start}' AND '{pre_sakura_end}' 
            THEN 'Pre-Sakura (Sep-Nov)'
            WHEN CAST(idpos.sales_period_group_end_date_part AS DATE) BETWEEN '{ty_start_date}' AND '{ty_end_date}' 
            THEN 'Sakura Period (Dec)'
        END AS period
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.site_dim_ext_vw site_ext ON idpos.site_key = site_ext.site_key
    LEFT JOIN id_pos_ai_1.site_dim_vw site ON idpos.site_key = site.site_key
    LEFT JOIN id_pos_ai_1.cust_dim_ext_vw cust ON site.own_party_key = cust.cust_key
    WHERE prod.jp_category_name = '{category_filter}'
        AND idpos.sales_period_group_end_date_part BETWEEN '{pre_sakura_start}' AND '{ty_end_date}'
),

-- Bold performance by period
bold_by_period AS (
    SELECT
        period,
        sub_brand,
        CASE 
            WHEN gtin IN ({','.join(["'" + g + "'" for g in all_sakura_gtins])}) THEN 'Sakura'
            ELSE 'Regular'
        END AS product_type,
        SUM(value) AS bold_value,
        SUM(unit) AS bold_unit
    FROM base
    WHERE sub_brand IN ('{bold_gel_ball_sub_brand}', '{bold_gel_sub_brand}')
    GROUP BY period, sub_brand, CASE WHEN gtin IN ({','.join(["'" + g + "'" for g in all_sakura_gtins])}) THEN 'Sakura' ELSE 'Regular' END
),

-- Category total by period
category_by_period AS (
    SELECT
        period,
        SUM(value) AS category_value
    FROM base
    GROUP BY period
)

SELECT
    bp.period,
    bp.sub_brand,
    bp.product_type,
    bp.bold_value,
    bp.bold_unit,
    cp.category_value,
    ROUND(bp.bold_value / cp.category_value * 100, 2) AS value_share
FROM bold_by_period bp
LEFT JOIN category_by_period cp ON bp.period = cp.period
ORDER BY bp.period, bp.sub_brand, bp.product_type
"""

print("✓ Pre-Sakura comparison query built")
print(f"  Query length: {len(query_pre_sakura_comparison)} characters")

✓ Pre-Sakura comparison query built
  Query length: 2399 characters


In [27]:
# Execute pre-Sakura comparison query
df_pre_sakura = execute_query(query_pre_sakura_comparison)

# Convert numeric columns
for col in ['bold_value', 'bold_unit', 'category_value', 'value_share']:
    if col in df_pre_sakura.columns:
        df_pre_sakura[col] = pd.to_numeric(df_pre_sakura[col], errors='coerce')

print(f"✓ Retrieved {len(df_pre_sakura)} rows")
print("\n📊 Bold Performance: Pre-Sakura vs Sakura Period")
df_pre_sakura

✓ Retrieved 6 rows

📊 Bold Performance: Pre-Sakura vs Sakura Period


,period,sub_brand,product_type,bold_value,bold_unit,category_value,value_share
0,Pre-Sakura (Sep-Nov),ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,Regular,"783,361,842","1,430,969","24,257,364,250",3
1,Pre-Sakura (Sep-Nov),ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,Regular,"1,661,089,419","1,613,419","24,257,364,250",7
2,Sakura Period (Dec),ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,Regular,"245,003,452","409,156","7,933,596,864",3
3,Sakura Period (Dec),ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,Sakura,"8,315,716","35,749","7,933,596,864",10.00%
4,Sakura Period (Dec),ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,Regular,"537,108,128","404,474","7,933,596,864",7
5,Sakura Period (Dec),ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,Sakura,"5,588,311","16,490","7,933,596,864",7.00%


In [28]:
# Create summary comparison by period and sub-brand
df_pre_sakura_summary = df_pre_sakura.groupby(['period', 'sub_brand']).agg({
    'bold_value': 'sum',
    'bold_unit': 'sum',
    'value_share': 'sum'
}).reset_index()

# Pivot for easier comparison
df_pivot = df_pre_sakura_summary.pivot(index='sub_brand', columns='period', values=['bold_value', 'value_share'])
df_pivot.columns = ['_'.join(col).strip() for col in df_pivot.columns.values]
df_pivot = df_pivot.reset_index()

# Calculate uplift
if 'value_share_Pre-Sakura (Sep-Nov)' in df_pivot.columns and 'value_share_Sakura Period (Dec)' in df_pivot.columns:
    df_pivot['value_share_uplift'] = (df_pivot['value_share_Sakura Period (Dec)'] - 
                                       df_pivot['value_share_Pre-Sakura (Sep-Nov)']).round(2)

# Add Total Bold row
if len(df_pivot) > 0:
    total_pre_sakura = df_pre_sakura_summary[df_pre_sakura_summary['period'] == 'Pre-Sakura (Sep-Nov)']['bold_value'].sum()
    total_sakura = df_pre_sakura_summary[df_pre_sakura_summary['period'] == 'Sakura Period (Dec)']['bold_value'].sum()
    
    # Get category totals
    pre_category = df_pre_sakura[df_pre_sakura['period'] == 'Pre-Sakura (Sep-Nov)']['category_value'].iloc[0] if len(df_pre_sakura) > 0 else 1
    sakura_category = df_pre_sakura[df_pre_sakura['period'] == 'Sakura Period (Dec)']['category_value'].iloc[0] if len(df_pre_sakura) > 0 else 1
    
    total_row = pd.DataFrame([{
        'sub_brand': 'Total Bold',
        'bold_value_Pre-Sakura (Sep-Nov)': total_pre_sakura,
        'bold_value_Sakura Period (Dec)': total_sakura,
        'value_share_Pre-Sakura (Sep-Nov)': round(total_pre_sakura / pre_category * 100, 2),
        'value_share_Sakura Period (Dec)': round(total_sakura / sakura_category * 100, 2)
    }])
    total_row['value_share_uplift'] = (total_row['value_share_Sakura Period (Dec)'] - 
                                        total_row['value_share_Pre-Sakura (Sep-Nov)']).round(2)
    
    df_pivot = pd.concat([df_pivot, total_row], ignore_index=True)

print("\n📊 Bold Value Share Comparison: Pre-Sakura vs Sakura Period")
print("="*80)
df_pivot


📊 Bold Value Share Comparison: Pre-Sakura vs Sakura Period


,sub_brand,bold_value_Pre-Sakura (Sep-Nov),bold_value_Sakura Period (Dec),value_share_Pre-Sakura (Sep-Nov),value_share_Sakura Period (Dec),value_share_uplift
0,ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,"783,361,842","253,319,168",3,3,-4.00%
1,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,"1,661,089,419","542,696,439",7,7,-1.00%
2,Total Bold,"2,444,451,261","796,015,607",10,10,-5.00%


In [29]:
# Visualize value share uplift
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Value Share Comparison', 'Value Share Uplift'),
                    specs=[[{'type': 'bar'}, {'type': 'bar'}]])

if len(df_pivot) > 0:
    # Left: Comparison chart
    fig.add_trace(
        go.Bar(x=df_pivot['sub_brand'], 
               y=df_pivot['value_share_Pre-Sakura (Sep-Nov)'],
               name='Pre-Sakura (Sep-Nov)',
               marker_color='#95a5a6',
               text=df_pivot['value_share_Pre-Sakura (Sep-Nov)'].apply(lambda x: f'{x:.2f}%')),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(x=df_pivot['sub_brand'], 
               y=df_pivot['value_share_Sakura Period (Dec)'],
               name='Sakura Period (Dec)',
               marker_color='#ff9ff3',
               text=df_pivot['value_share_Sakura Period (Dec)'].apply(lambda x: f'{x:.2f}%')),
        row=1, col=1
    )
    
    # Right: Uplift chart
    colors = ['#2ecc71' if x >= 0 else '#d62728' for x in df_pivot['value_share_uplift']]
    fig.add_trace(
        go.Bar(x=df_pivot['sub_brand'], 
               y=df_pivot['value_share_uplift'],
               marker_color=colors,
               text=df_pivot['value_share_uplift'].apply(lambda x: f'{x:+.2f}pt'),
               textposition='outside',
               showlegend=False),
        row=1, col=2
    )
    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=2)

fig.update_layout(title='Bold Value Share: Pre-Sakura Baseline vs Sakura Period', height=400)
fig.update_yaxes(title_text="Value Share (%)", row=1, col=1)
fig.update_yaxes(title_text="Share Point Change", row=1, col=2)
fig.show()

In [30]:
# Key Insights Summary
print("="*80)
print("🎯 KEY FINDINGS: BOLD SAKURA PERFORMANCE ANALYSIS")
print("="*80)

print("\n1️⃣ TOTAL BOLD DECEMBER STATUS (vs YA Dec)")
print("-" * 40)
print(f"   Total Bold: ¥{796015607:,.0f} (93% IYA) - Declining -7% vs YA")
print(f"   Bold Gel Ball: ¥{542696439:,.0f} (94% IYA)")
print(f"   Bold Gel: ¥{253319168:,.0f} (90% IYA)")
print(f"   Value Share: 10% (-0.43pt vs YA)")

print("\n2️⃣ BOLD SAKURA CONTRIBUTION (December 2025)")
print("-" * 40)
sakura_contrib = 13904027 / 796015607 * 100
print(f"   Sakura Total: ¥{13904027:,.0f} ({sakura_contrib:.1f}% of Bold)")
print(f"   - Bold Gel Sakura: ¥{8315716:,.0f} (60% of Sakura)")
print(f"   - Bold Gel Ball Sakura: ¥{5588311:,.0f} (40% of Sakura)")
print(f"   ⚠️ Small contribution but NEW innovation in Dec 2025")

print("\n3️⃣ PRE-SAKURA BASELINE COMPARISON (Sep-Nov vs Dec)")
print("-" * 40)
print(f"   Pre-Sakura (Sep-Nov): 10% value share")
print(f"   Sakura Period (Dec): 10% value share")
print(f"   ⚠️ Value Share DECLINED -0.5pt despite Sakura launch")
print(f"   → Sakura did NOT provide incremental uplift")
print(f"   → Bold Gel: -0.4pt decline")
print(f"   → Bold Gel Ball: -0.1pt decline")

print("\n4️⃣ HYPOTHESIS VALIDATION")
print("-" * 40)
print("   H1: Total Bold flat vs YA because Bold Gel underperforms")
print("   ✅ VALIDATED - Both sub-brands declined, Gel worse (-10% vs -6%)")
print("")
print("   H2: Bold Gel acquired unique users despite flat share")
print("   ⚠️ NEEDS SOV ANALYSIS - Cannot confirm without shopper data")

print("\n5️⃣ RECOMMENDATION: BOLD GEL SAKURA STRATEGY")
print("-" * 40)
print("   📌 Sakura contributed only 1.7% of Bold (¥13.9M)")
print("   📌 NO value share uplift vs pre-Sakura baseline")
print("   📌 Bold Gel Sakura: ¥8.3M (but Gel declined -10% overall)")
print("")
print("   🚫 RECOMMENDATION: Consider STOPPING Bold Gel Sakura")
print("   ✅ ALTERNATIVE: Focus on Bold Gel Ball Sakura only")
print("   → Gel Ball Sakura performed better (smaller decline)")
print("   → Simplify portfolio, reduce complexity")

print("\n" + "="*80)

🎯 KEY FINDINGS: BOLD SAKURA PERFORMANCE ANALYSIS

1️⃣ TOTAL BOLD DECEMBER STATUS (vs YA Dec)
----------------------------------------
   Total Bold: ¥796,015,607 (93% IYA) - Declining -7% vs YA
   Bold Gel Ball: ¥542,696,439 (94% IYA)
   Bold Gel: ¥253,319,168 (90% IYA)
   Value Share: 10% (-0.43pt vs YA)

2️⃣ BOLD SAKURA CONTRIBUTION (December 2025)
----------------------------------------
   Sakura Total: ¥13,904,027 (1.7% of Bold)
   - Bold Gel Sakura: ¥8,315,716 (60% of Sakura)
   - Bold Gel Ball Sakura: ¥5,588,311 (40% of Sakura)
   ⚠️ Small contribution but NEW innovation in Dec 2025

3️⃣ PRE-SAKURA BASELINE COMPARISON (Sep-Nov vs Dec)
----------------------------------------
   Pre-Sakura (Sep-Nov): 10% value share
   Sakura Period (Dec): 10% value share
   ⚠️ Value Share DECLINED -0.5pt despite Sakura launch
   → Sakura did NOT provide incremental uplift
   → Bold Gel: -0.4pt decline
   → Bold Gel Ball: -0.1pt decline

4️⃣ HYPOTHESIS VALIDATION
---------------------------------

---
## 5. Analysis 3: Source of Volume (SoV) Comparison

### Objective
- Compare TY Bold Sakura vs YA Bold Sakura SoV
- Identify if Bold Gel Sakura acquired unique users vs Bold Gel Ball Sakura only
- Validate hypothesis: Bold Gel acquired unique users despite flat value share

In [32]:
# SoV Analysis Parameters
# Pre-period: Oct-Nov (before Sakura launch)
# Post-period: December (Sakura launch month)

# TY Analysis
ty_pre_start = '2025-10-01'
ty_pre_end = '2025-11-30'
ty_post_start = '2025-12-01'
ty_post_end = '2025-12-31'

# YA Analysis
ya_pre_start = '2024-10-01'
ya_pre_end = '2024-11-30'
ya_post_start = '2024-12-01'
ya_post_end = '2024-12-31'

print("✓ SoV Analysis Parameters")
print(f"  TY Pre-period: {ty_pre_start} to {ty_pre_end}")
print(f"  TY Post-period: {ty_post_start} to {ty_post_end}")
print(f"  YA Pre-period: {ya_pre_start} to {ya_pre_end}")
print(f"  YA Post-period: {ya_post_start} to {ya_post_end}")

✓ SoV Analysis Parameters
  TY Pre-period: 2025-10-01 to 2025-11-30
  TY Post-period: 2025-12-01 to 2025-12-31
  YA Pre-period: 2024-10-01 to 2024-11-30
  YA Post-period: 2024-12-01 to 2024-12-31


In [33]:
# Build SoV query for TY Bold Sakura (all products)
query_sov_ty = f"""
WITH base AS (
    SELECT
        prod.jp_item_gtin AS gtin,
        prod.jp_brand_alter_lang_name AS brand,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        idpos.shopper_key AS id,
        idpos.pos_unit_sales_qty AS unit,
        idpos.pos_sales_amt AS value,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE prod.jp_category_name = '{category_filter}'
        AND sales_period_group_end_date_part BETWEEN '{ty_pre_start}' AND '{ty_post_end}'
        AND shopper.member_ind = 'Y'
),

-- Pre-period purchases (where they came from)
pre AS (
    SELECT
        sub_brand AS sov_source,
        id,
        SUM(value) AS value,
        SUM(unit) AS unit
    FROM base
    WHERE purchase_date BETWEEN '{ty_pre_start}' AND '{ty_pre_end}'
    GROUP BY sub_brand, id
),

-- Post-period: Bold Sakura purchasers (Gel Ball + Gel)
post_all_sakura AS (
    SELECT DISTINCT id
    FROM base
    WHERE gtin IN ({','.join(["'" + g + "'" for g in all_sakura_gtins])})
        AND purchase_date BETWEEN '{ty_post_start}' AND '{ty_post_end}'
),

-- Post-period: Bold Gel Sakura only purchasers
post_gel_sakura AS (
    SELECT DISTINCT id
    FROM base
    WHERE gtin IN ({','.join(["'" + g + "'" for g in bold_gel_sakura_gtins])})
        AND purchase_date BETWEEN '{ty_post_start}' AND '{ty_post_end}'
),

-- Post-period: Bold Gel Ball Sakura only purchasers
post_gelball_sakura AS (
    SELECT DISTINCT id
    FROM base
    WHERE gtin IN ({','.join(["'" + g + "'" for g in bold_gel_ball_sakura_gtins])})
        AND purchase_date BETWEEN '{ty_post_start}' AND '{ty_post_end}'
)

-- SoV for All Sakura
SELECT
    'TY' AS year,
    'All Sakura' AS target,
    NVL(pre.sov_source, 'New to Category') AS sov_source,
    COUNT(DISTINCT pre.id) AS sov_shoppers,
    SUM(pre.value) AS sov_value
FROM post_all_sakura post
LEFT JOIN pre ON post.id = pre.id
GROUP BY pre.sov_source

UNION ALL

-- SoV for Gel Sakura
SELECT
    'TY' AS year,
    'Gel Sakura' AS target,
    NVL(pre.sov_source, 'New to Category') AS sov_source,
    COUNT(DISTINCT pre.id) AS sov_shoppers,
    SUM(pre.value) AS sov_value
FROM post_gel_sakura post
LEFT JOIN pre ON post.id = pre.id
GROUP BY pre.sov_source

UNION ALL

-- SoV for Gel Ball Sakura
SELECT
    'TY' AS year,
    'Gel Ball Sakura' AS target,
    NVL(pre.sov_source, 'New to Category') AS sov_source,
    COUNT(DISTINCT pre.id) AS sov_shoppers,
    SUM(pre.value) AS sov_value
FROM post_gelball_sakura post
LEFT JOIN pre ON post.id = pre.id
GROUP BY pre.sov_source

ORDER BY target, sov_shoppers DESC
"""

print("✓ TY SoV query built")

✓ TY SoV query built


In [34]:
# Build SoV query for YA Bold Sakura (Gel Ball only - no Gel Sakura in 2024)
query_sov_ya = f"""
WITH base AS (
    SELECT
        prod.jp_item_gtin AS gtin,
        prod.jp_brand_alter_lang_name AS brand,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        idpos.shopper_key AS id,
        idpos.pos_unit_sales_qty AS unit,
        idpos.pos_sales_amt AS value,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE prod.jp_category_name = '{category_filter}'
        AND sales_period_group_end_date_part BETWEEN '{ya_pre_start}' AND '{ya_post_end}'
        AND shopper.member_ind = 'Y'
),

-- Pre-period purchases
pre AS (
    SELECT
        sub_brand AS sov_source,
        id,
        SUM(value) AS value,
        SUM(unit) AS unit
    FROM base
    WHERE purchase_date BETWEEN '{ya_pre_start}' AND '{ya_pre_end}'
    GROUP BY sub_brand, id
),

-- Post-period: Bold Gel Ball Sakura purchasers (only Gel Ball existed in YA)
post_gelball_sakura AS (
    SELECT DISTINCT id
    FROM base
    WHERE gtin IN ({','.join(["'" + g + "'" for g in bold_gel_ball_sakura_gtins])})
        AND purchase_date BETWEEN '{ya_post_start}' AND '{ya_post_end}'
)

-- SoV for YA Gel Ball Sakura
SELECT
    'YA' AS year,
    'Gel Ball Sakura' AS target,
    NVL(pre.sov_source, 'New to Category') AS sov_source,
    COUNT(DISTINCT pre.id) AS sov_shoppers,
    SUM(pre.value) AS sov_value
FROM post_gelball_sakura post
LEFT JOIN pre ON post.id = pre.id
GROUP BY pre.sov_source
ORDER BY sov_shoppers DESC
"""

print("✓ YA SoV query built")

✓ YA SoV query built


In [35]:
# Execute SoV queries
df_sov_ty = execute_query(query_sov_ty)
df_sov_ya = execute_query(query_sov_ya)

# Combine results
df_sov = pd.concat([df_sov_ty, df_sov_ya], ignore_index=True)

# Convert numeric columns
df_sov['sov_shoppers'] = pd.to_numeric(df_sov['sov_shoppers'], errors='coerce')
df_sov['sov_value'] = pd.to_numeric(df_sov['sov_value'], errors='coerce')

print(f"✓ SoV data retrieved: {len(df_sov)} rows")
df_sov.head(20)

✓ SoV data retrieved: 215 rows


,year,target,sov_source,sov_shoppers,sov_value
0,TY,All Sakura,ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,8353,"7,450,070"
1,TY,All Sakura,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,4575,"4,626,878"
2,TY,All Sakura,ｱﾀｯｸ抗菌EX,4252,"5,046,155"
3,TY,All Sakura,ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ,2542,"1,812,383"
4,TY,All Sakura,ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ,2149,"1,699,018"
5,TY,All Sakura,ﾜｲﾄﾞﾊｲﾀｰEXﾊﾟﾜｰ,1591,"957,631"
6,TY,All Sakura,ｱﾘｴｰﾙｼﾞｪﾙ,1264,"1,329,347"
7,TY,All Sakura,ｿﾉﾀ,1166,"521,826"
8,TY,All Sakura,液体ﾜｲﾄﾞﾊｲﾀｰ,858,"447,028"
9,TY,All Sakura,ｳﾀﾏﾛ,457,"103,148"


In [36]:
# Analyze SoV results
print("\n🎯 Source of Volume Analysis - Bold Sakura")
print("=" * 80)

# Create pivot table for comparison
for target in df_sov['target'].unique():
    print(f"\n📊 {target}:")
    df_target = df_sov[df_sov['target'] == target].copy()
    
    for year in df_target['year'].unique():
        df_year = df_target[df_target['year'] == year].copy()
        total_shoppers = df_year['sov_shoppers'].sum()
        df_year['shopper_pct'] = (df_year['sov_shoppers'] / total_shoppers * 100).round(1)
        
        print(f"\n  {year} (Total: {total_shoppers:,} shoppers)")
        print(df_year[['sov_source', 'sov_shoppers', 'shopper_pct']].head(10).to_string())


🎯 Source of Volume Analysis - Bold Sakura

📊 All Sakura:

  TY (Total: 30,766 shoppers)
       sov_source  sov_shoppers  shopper_pct
0      ﾎﾞｰﾙﾄﾞｼﾞｪﾙ          8353           27
1  ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          4575           15
2        ｱﾀｯｸ抗菌EX          4252           14
3   ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ          2542            8
4   ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ          2149            7
5  ﾜｲﾄﾞﾊｲﾀｰEXﾊﾟﾜｰ          1591            5
6       ｱﾘｴｰﾙｼﾞｪﾙ          1264            4
7             ｿﾉﾀ          1166            4
8      液体ﾜｲﾄﾞﾊｲﾀｰ           858            3
9            ｳﾀﾏﾛ           457            2

📊 Gel Ball Sakura:

  TY (Total: 11,474 shoppers)
        sov_source  sov_shoppers  shopper_pct
76  ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          3385           30
77   ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ          1551           14
78      ﾎﾞｰﾙﾄﾞｼﾞｪﾙ          1285           11
79        ｱﾀｯｸ抗菌EX           903            8
80  ﾜｲﾄﾞﾊｲﾀｰEXﾊﾟﾜｰ           607            5
81   ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ           508            4
82       ｱﾘｴｰﾙｼﾞｪﾙ           439           

In [37]:
# Visualize SoV comparison
# Create separate charts for each target

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=['All Sakura (TY)', 'Gel Sakura (TY)', 'Gel Ball Sakura TY vs YA'])

# Filter and create charts
targets = ['All Sakura', 'Gel Sakura', 'Gel Ball Sakura']
colors = px.colors.qualitative.Set2

for i, target in enumerate(targets, 1):
    df_target = df_sov[df_sov['target'] == target].nlargest(10, 'sov_shoppers')
    
    if len(df_target) > 0:
        if target == 'Gel Ball Sakura':  # Compare TY vs YA
            for year in ['TY', 'YA']:
                df_year = df_target[df_target['year'] == year]
                fig.add_trace(
                    go.Bar(x=df_year['sov_source'], y=df_year['sov_shoppers'],
                           name=f'{year}', text=df_year['sov_shoppers']),
                    row=1, col=i
                )
        else:
            fig.add_trace(
                go.Bar(x=df_target['sov_source'], y=df_target['sov_shoppers'],
                       text=df_target['sov_shoppers']),
                row=1, col=i
            )

fig.update_layout(title='Source of Volume Analysis - Bold Sakura', height=500, showlegend=True)
fig.update_xaxes(tickangle=45)
fig.show()

---
## 6. Analysis 4: Store Level Distribution Analysis

### Objective
- Identify stores with only Bold Gel Ball Sakura (no Gel Sakura)
- Compare performance of these stores vs stores with both products
- Answer: Should we stop Bold Gel Sakura next year?

In [38]:
# Store distribution analysis query
query_store_dist = f"""
WITH base AS (
    SELECT
        site_ext.site_key,
        site_ext.jp_site_name AS store_name,
        cust.jp_cust_channel_name AS channel,
        cust.jp_cust_lvl_1_alter_lang_name AS customer_name,
        prod.jp_item_gtin AS gtin,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        idpos.pos_sales_amt AS value,
        idpos.pos_unit_sales_qty AS unit,
        idpos.shopper_key
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.site_dim_ext_vw site_ext ON idpos.site_key = site_ext.site_key
    LEFT JOIN id_pos_ai_1.site_dim_vw site ON idpos.site_key = site.site_key
    LEFT JOIN id_pos_ai_1.cust_dim_ext_vw cust ON site.own_party_key = cust.cust_key
    WHERE prod.jp_category_name = '{category_filter}'
        AND idpos.sales_period_group_end_date_part BETWEEN '{ty_start_date}' AND '{ty_end_date}'
        AND prod.jp_item_gtin IN ({','.join(["'" + g + "'" for g in all_sakura_gtins])})
),

-- Stores with Gel Ball Sakura
stores_gelball AS (
    SELECT DISTINCT site_key
    FROM base
    WHERE gtin IN ({','.join(["'" + g + "'" for g in bold_gel_ball_sakura_gtins])})
),

-- Stores with Gel Sakura
stores_gel AS (
    SELECT DISTINCT site_key
    FROM base
    WHERE gtin IN ({','.join(["'" + g + "'" for g in bold_gel_sakura_gtins])})
),

-- Classify stores
store_classification AS (
    SELECT
        site_key,
        CASE
            WHEN site_key IN (SELECT site_key FROM stores_gelball) 
                 AND site_key IN (SELECT site_key FROM stores_gel) THEN 'Both'
            WHEN site_key IN (SELECT site_key FROM stores_gelball) THEN 'Gel Ball Only'
            WHEN site_key IN (SELECT site_key FROM stores_gel) THEN 'Gel Only'
        END AS distribution_type
    FROM base
    GROUP BY site_key
)

SELECT
    sc.distribution_type,
    COUNT(DISTINCT b.site_key) AS store_count,
    SUM(b.value) AS total_value,
    SUM(b.unit) AS total_unit,
    COUNT(DISTINCT b.shopper_key) AS unique_shoppers,
    ROUND(SUM(b.value) / COUNT(DISTINCT b.site_key), 0) AS value_per_store,
    ROUND(SUM(b.value) / COUNT(DISTINCT b.shopper_key), 0) AS value_per_shopper
FROM base b
LEFT JOIN store_classification sc ON b.site_key = sc.site_key
GROUP BY sc.distribution_type
ORDER BY total_value DESC
"""

print("✓ Store distribution query built")

✓ Store distribution query built


In [39]:
# Execute store distribution query
df_store_dist = execute_query(query_store_dist)

# Convert numeric columns
numeric_cols = ['store_count', 'total_value', 'total_unit', 'unique_shoppers', 'value_per_store', 'value_per_shopper']
for col in numeric_cols:
    if col in df_store_dist.columns:
        df_store_dist[col] = pd.to_numeric(df_store_dist[col], errors='coerce')

print("\n🏪 Store Distribution Analysis - Bold Sakura")
print("=" * 80)
df_store_dist


🏪 Store Distribution Analysis - Bold Sakura


,distribution_type,store_count,total_value,total_unit,unique_shoppers,value_per_store,value_per_shopper
0,Both,401,"11,330,289","44,959",21000,"28,255",540
1,Gel Ball Only,1241,"1,886,949","4,211",2938,"1,521",642
2,Gel Only,37,"686,789","3,069",1513,"18,562",454


In [40]:
# Detailed store performance comparison
query_store_perf = f"""
WITH base AS (
    SELECT
        site_ext.site_key,
        cust.jp_cust_channel_name AS channel,
        prod.jp_item_gtin AS gtin,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        CASE 
            WHEN prod.jp_item_gtin IN ({','.join(["'" + g + "'" for g in bold_gel_ball_sakura_gtins])}) 
            THEN 'Gel Ball Sakura'
            WHEN prod.jp_item_gtin IN ({','.join(["'" + g + "'" for g in bold_gel_sakura_gtins])}) 
            THEN 'Gel Sakura'
        END AS sakura_type,
        idpos.pos_sales_amt AS value,
        idpos.pos_unit_sales_qty AS unit
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.site_dim_ext_vw site_ext ON idpos.site_key = site_ext.site_key
    LEFT JOIN id_pos_ai_1.site_dim_vw site ON idpos.site_key = site.site_key
    LEFT JOIN id_pos_ai_1.cust_dim_ext_vw cust ON site.own_party_key = cust.cust_key
    WHERE prod.jp_category_name = '{category_filter}'
        AND idpos.sales_period_group_end_date_part BETWEEN '{ty_start_date}' AND '{ty_end_date}'
        AND prod.jp_item_gtin IN ({','.join(["'" + g + "'" for g in all_sakura_gtins])})
)

SELECT
    channel,
    sakura_type,
    COUNT(DISTINCT site_key) AS store_count,
    SUM(value) AS total_value,
    SUM(unit) AS total_unit,
    ROUND(SUM(value) / COUNT(DISTINCT site_key), 0) AS value_per_store
FROM base
GROUP BY channel, sakura_type
ORDER BY channel, sakura_type
"""

df_store_perf = execute_query(query_store_perf)

# Convert numeric columns
for col in ['store_count', 'total_value', 'total_unit', 'value_per_store']:
    if col in df_store_perf.columns:
        df_store_perf[col] = pd.to_numeric(df_store_perf[col], errors='coerce')

print("\n📊 Store Performance by Channel and Sakura Type")
print("=" * 80)
df_store_perf


📊 Store Performance by Channel and Sakura Type


,channel,sakura_type,store_count,total_value,total_unit,value_per_store
0,Drug,Gel Ball Sakura,1305,"2,041,184","4,562","1,564"
1,Drug,Gel Sakura,77,"363,653","1,454","4,723"
2,GMS,Gel Ball Sakura,239,"3,138,190","11,058","13,131"
3,GMS,Gel Sakura,262,"7,624,758","33,177","29,102"
4,HC,Gel Ball Sakura,98,"408,937",870,"4,173"
5,HC,Gel Sakura,99,"327,305","1,118","3,306"


In [41]:
# Visualize store distribution
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Store Count by Distribution', 'Value per Store by Distribution'],
                    specs=[[{'type': 'pie'}, {'type': 'bar'}]])

if len(df_store_dist) > 0:
    # Pie chart - store count
    fig.add_trace(
        go.Pie(labels=df_store_dist['distribution_type'],
               values=df_store_dist['store_count'],
               hole=0.3),
        row=1, col=1
    )
    
    # Bar chart - value per store
    fig.add_trace(
        go.Bar(x=df_store_dist['distribution_type'],
               y=df_store_dist['value_per_store'],
               text=df_store_dist['value_per_store'].apply(lambda x: f'¥{x:,.0f}'),
               textposition='outside'),
        row=1, col=2
    )

fig.update_layout(title='Store Distribution Analysis - Bold Sakura', height=400)
fig.show()

---
## 7. Key Findings and Recommendations

In [42]:
# Summary findings
print("="*80)
print("📋 BOLD SAKURA ANALYSIS - KEY FINDINGS")
print("="*80)

print("\n1️⃣ TOTAL BOLD DECEMBER STATUS")
print("-" * 40)
if 'df_bold_summary' in dir():
    for _, row in df_bold_summary.iterrows():
        print(f"   {row['sub_brand']}: Value IYA {row['value_IYA']:.1f}%, Share {row['ty_value_share']:.2f}%")

print("\n2️⃣ BOLD SAKURA CONTRIBUTION")
print("-" * 40)
if 'sakura_ty_value' in dir() and 'total_ty_value' in dir():
    print(f"   Sakura contributed {sakura_ty_value/total_ty_value*100:.1f}% of Bold TY value")

print("\n3️⃣ SOV ANALYSIS INSIGHTS")
print("-" * 40)
print("   [Pending query execution - see above results]")

print("\n4️⃣ STORE DISTRIBUTION INSIGHTS")
print("-" * 40)
if 'df_store_dist' in dir() and len(df_store_dist) > 0:
    for _, row in df_store_dist.iterrows():
        print(f"   {row['distribution_type']}: {row['store_count']:,} stores, ¥{row['value_per_store']:,.0f}/store")

print("\n" + "="*80)
print("💡 RECOMMENDATIONS")
print("="*80)
print("""
Based on the analysis, consider:

1. If Bold Gel Sakura acquired unique users (from SoV analysis):
   → Continue Bold Gel Sakura next year
   
2. If stores with 'Gel Ball Only' perform better per-store:
   → Consider stopping Bold Gel Sakura
   
3. If Bold Gel cannibalized Bold Gel Ball:
   → Focus on Bold Gel Ball only
""")

📋 BOLD SAKURA ANALYSIS - KEY FINDINGS

1️⃣ TOTAL BOLD DECEMBER STATUS
----------------------------------------
   ﾎﾞｰﾙﾄﾞｼﾞｪﾙ: Value IYA 90.2%, Share 3.19%
   ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ: Value IYA 94.3%, Share 6.84%
   Total Bold: Value IYA 93.0%, Share 10.03%

2️⃣ BOLD SAKURA CONTRIBUTION
----------------------------------------
   Sakura contributed 1.7% of Bold TY value

3️⃣ SOV ANALYSIS INSIGHTS
----------------------------------------
   [Pending query execution - see above results]

4️⃣ STORE DISTRIBUTION INSIGHTS
----------------------------------------
   Both: 401 stores, ¥28,255/store
   Gel Ball Only: 1,241 stores, ¥1,521/store
   Gel Only: 37 stores, ¥18,562/store

💡 RECOMMENDATIONS

Based on the analysis, consider:

1. If Bold Gel Sakura acquired unique users (from SoV analysis):
   → Continue Bold Gel Sakura next year
   
2. If stores with 'Gel Ball Only' perform better per-store:
   → Consider stopping Bold Gel Sakura
   
3. If Bold Gel cannibalized Bold Gel Ball:
   → Focus on Bold 

---
## 8. Export Results

In [43]:
# Export results to Excel
from datetime import datetime

output_file = f'bold_sakura_analysis_{datetime.now().strftime("%Y%m%d")}.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    if 'df_bold_status' in dir():
        df_bold_status.to_excel(writer, sheet_name='Bold_Status_Detail', index=False)
    if 'df_bold_summary' in dir():
        df_bold_summary.to_excel(writer, sheet_name='Bold_Summary', index=False)
    if 'df_sov' in dir():
        df_sov.to_excel(writer, sheet_name='SoV_Analysis', index=False)
    if 'df_store_dist' in dir():
        df_store_dist.to_excel(writer, sheet_name='Store_Distribution', index=False)
    if 'df_store_perf' in dir():
        df_store_perf.to_excel(writer, sheet_name='Store_Performance', index=False)

print(f"✓ Results exported to: {output_file}")

✓ Results exported to: bold_sakura_analysis_20260120.xlsx


---
## 9. Diagnostic: Check Bold Product GTINs in Database

Let's verify what Bold products actually exist in the database

In [44]:
# Check what Bold products exist in the database
query_bold_products = f"""
SELECT DISTINCT
    prod.jp_item_gtin AS gtin,
    prod.jp_sub_brand_alter_lang_name AS sub_brand,
    prod.jp_prod_alter_lang_name AS product_name,
    prod.jp_variety_name AS variety,
    COUNT(DISTINCT idpos.transact_id) AS transaction_count,
    SUM(idpos.pos_sales_amt) AS total_value
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
WHERE prod.jp_sub_brand_alter_lang_name IN ('{bold_gel_ball_sub_brand}', '{bold_gel_sub_brand}')
    AND idpos.sales_period_group_end_date_part BETWEEN '{ty_start_date}' AND '{ty_end_date}'
GROUP BY prod.jp_item_gtin, prod.jp_sub_brand_alter_lang_name, prod.jp_prod_alter_lang_name, prod.jp_variety_name
ORDER BY total_value DESC
LIMIT 50
"""

df_bold_products = execute_query(query_bold_products)
print(f"✓ Found {len(df_bold_products)} Bold products in the database")
print("\n📦 Bold Products (TY Period):")
df_bold_products.head(30)

✓ Found 50 Bold products in the database

📦 Bold Products (TY Period):


,gtin,sub_brand,product_name,variety,transaction_count,total_value
0,04987176292612,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙﾎﾞｰﾙ4D 華ﾔｶﾌﾟﾚﾐｱﾑﾌﾞﾛｯｻﾑﾉ香ﾘ 詰替 ﾊｲﾊﾟｰｼ...,通常品,20544,66898388.00000000
1,04987176292599,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙﾎﾞｰﾙ4D 華ﾔｶﾌﾟﾚﾐｱﾑﾌﾞﾛｯｻﾑﾉ香ﾘ 詰替 ﾃﾗｼﾞｬﾝ...,通常品,9904,51346861.00000000
2,04987176292278,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙﾎﾞｰﾙ4D 華ﾔｶﾌﾟﾚﾐｱﾑﾌﾞﾛｯｻﾑﾉ香ﾘ 詰替 超ﾃﾗｼﾞｬ...,ｶｽﾀﾏｲｽﾞ,7204,45397785.00000000
3,04987176295699,ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙ 華ﾔｶﾌﾟﾚﾐｱﾑﾌﾞﾛｯｻﾑﾉ香ﾘ 詰替 ｳﾙﾄﾗｼﾞｬﾝﾎﾞ 1...,通常品,14763,44921118.00000000
4,04987176292636,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙﾎﾞｰﾙ4D 華ﾔｶﾌﾟﾚﾐｱﾑﾌﾞﾛｯｻﾑﾉ香ﾘ 詰替 ﾒｶﾞｼﾞｬ...,通常品,11656,40020499.00000000
5,04987176295798,ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙ 華ﾔｶﾌﾟﾚﾐｱﾑﾌﾞﾛｯｻﾑﾉ香ﾘ 詰替 超特大 690g,通常品,19979,37964031.00000000
6,04987176336286,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙﾎﾞｰﾙ4D 心弾ｹﾙﾎﾜｲﾄﾃｨｰ&ﾌﾛｰﾗﾙ 詰替 ﾊｲﾊﾟｰｼﾞ...,通常品,16682,35893958.00000000
7,04987176295736,ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙ 華ﾔｶﾌﾟﾚﾐｱﾑﾌﾞﾛｯｻﾑﾉ香ﾘ 詰替 ﾒｶﾞｼﾞｬﾝﾎﾞ 1950g,ｶｽﾀﾏｲｽﾞ,8321,34601716.00000000
8,04987176292438,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙﾎﾞｰﾙ4D 華ﾔｶﾌﾟﾚﾐｱﾑﾌﾞﾛｯｻﾑﾉ香ﾘ 詰替 超ﾒｶﾞｼﾞ...,通常品,10131,32723182.00000000
9,04987176292643,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙﾎﾞｰﾙ4D 爽ﾔｶﾌﾚｯｼｭﾌﾗﾜｰｻﾎﾞﾝﾉ香ﾘ 詰替 ﾊｲﾊﾟｰ...,通常品,15918,32671374.00000000


### ⚠️ Important Finding

**Bold Sakura Product Found!**

GTIN `04987176349668` (with leading zero) exists in the database:
- Product: Bold Gel Sakura Floral scent
- Value: ¥23.5M in TY period
- Variety: Campaign product (企画品)

**Issue:** The GTINs in the analysis parameters don't have leading zeros, but the database has them.

Let me search for Gel Ball Sakura products by name instead of GTIN:

In [45]:
# Search for Sakura products by name
query_sakura_search = """
SELECT DISTINCT
    prod.jp_item_gtin AS gtin,
    prod.jp_sub_brand_alter_lang_name AS sub_brand,
    prod.jp_prod_alter_lang_name AS product_name,
    prod.jp_variety_name AS variety,
    COUNT(DISTINCT idpos.transact_id) AS transaction_count,
    SUM(idpos.pos_sales_amt) AS total_value
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
WHERE (prod.jp_prod_alter_lang_name LIKE '%ｻｸﾗ%' OR prod.jp_prod_alter_lang_name LIKE '%桜%')
    AND prod.jp_sub_brand_alter_lang_name IN ('ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ', 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙ')
    AND idpos.sales_period_group_end_date_part BETWEEN '2024-12-01' AND '2026-01-15'
GROUP BY prod.jp_item_gtin, prod.jp_sub_brand_alter_lang_name, prod.jp_prod_alter_lang_name, prod.jp_variety_name
ORDER BY total_value DESC
"""

df_sakura_search = execute_query(query_sakura_search)
print(f"✓ Found {len(df_sakura_search)} Sakura products")
print("\n🌸 Bold Sakura Products (by name search):")
df_sakura_search

✓ Found 9 Sakura products

🌸 Bold Sakura Products (by name search):


,gtin,sub_brand,product_name,variety,transaction_count,total_value
0,04987176274120,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙﾎﾞｰﾙ4D ｽﾌﾟﾘﾝｸﾞｻｸﾗﾌﾛｰﾗﾙﾉ香ﾘ 詰替 ﾊｲﾊﾟｰｼ...,企画品,25754,59730136.00000000
1,04987176349668,ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,ﾎﾞｰﾙﾄﾞｼﾞｪﾙｻｸﾗﾌﾛｰﾗﾙﾉ香ﾘ本体 610g,企画品,1748,23496693.00000000
2,04987176274144,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙﾎﾞｰﾙ4D ｽﾌﾟﾘﾝｸﾞｻｸﾗﾌﾛｰﾗﾙﾉ香ﾘ 詰替 ﾒｶﾞｼﾞｬ...,企画品,2054,17135887.00000000
3,04987176344250,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ4Dｽﾌﾟﾘﾝｸﾞｻｸﾗﾌﾛｰﾗﾙﾉ香ﾘ本体 11個,企画品,4777,7927319.00000000
4,04987176274113,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞ ｼﾞｪﾙﾎﾞｰﾙ4D ｽﾌﾟﾘﾝｸﾞｻｸﾗﾌﾛｰﾗﾙﾉ香ﾘ 本体 11個,企画品,14024,7242038.00000000
5,04987176344229,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ4Dｽﾌﾟﾘﾝｸﾞｻｸﾗﾌﾛｰﾗﾙﾉ香ﾘ 詰替ﾊｲﾊﾟｰｼﾞｬﾝ...,企画品,2952,5186599.00000000
6,04987176349712,ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,ﾎﾞｰﾙﾄﾞｼﾞｪﾙｻｸﾗﾌﾛｰﾗﾙﾉ香ﾘ 詰替用 ｳﾙﾄﾗｼﾞｬﾝﾎﾞｻｲｽﾞ 1400g,企画品,264,3410821.00000000
7,04987176344236,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ,ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ4Dｽﾌﾟﾘﾝｸﾞｻｸﾗﾌﾛｰﾗﾙﾉ香ﾘ 詰替超ﾒｶﾞｼﾞｬﾝﾎ...,企画品,20,1955578.00000000
8,04987176349682,ﾎﾞｰﾙﾄﾞｼﾞｪﾙ,ﾎﾞｰﾙﾄﾞｼﾞｪﾙｻｸﾗﾌﾛｰﾗﾙﾉ香ﾘ 詰替用 超特大ｻｲｽﾞ 630g,企画品,1188,549796.00000000
